In [7]:
import os
import numpy as np
import pandas as pd

# =========================================================
# 1. Load Clean Datasets
# =========================================================
loan = pd.read_csv('../cleanData/clean_loan.csv', sep=';')
trans = pd.read_csv('../cleanData/clean_trans.csv', sep=';', low_memory=False)

# Normalize join keys and date formats
loan['account_id'] = loan['account_id'].astype(str)
loan['loan_id'] = loan['loan_id'].astype(str)
trans['account_id'] = trans['account_id'].astype(str)
trans['trans_id'] = trans['trans_id'].astype(str)

loan['loan_date'] = pd.to_datetime(loan['date'], errors='coerce')
trans['trans_date'] = pd.to_datetime(trans['date'], errors='coerce')

# Standardize string fields
trans['type'] = trans['type'].astype(str).str.strip().str.lower()
trans['operation'] = trans['operation'].astype(str).str.strip().str.lower()
trans['k_symbol'] = trans['k_symbol'].astype(str).str.strip().str.lower()

# =========================================================
# 2. Filter Pre-Loan Transactions
# =========================================================
loan_cols = [
    'account_id',
    'loan_id',
    'date',
    'amount',
    'duration',
    'payments',
    'status',
    'target',
]

trans_loan = pd.merge(
    trans,
    loan[loan_cols],
    on='account_id',
    how='inner',
    suffixes=('_trans', '_loan'),
)

# Standardize and force datetime types post-merge
if 'date_loan' in trans_loan.columns:
  trans_loan['loan_date'] = pd.to_datetime(
      trans_loan['date_loan'], errors='coerce'
  )
elif 'date' in trans_loan.columns:
  trans_loan['loan_date'] = pd.to_datetime(
      trans_loan['date'], errors='coerce'
  )

trans_loan['trans_date'] = pd.to_datetime(
    trans_loan['trans_date'], errors='coerce'
  )

# Strictly before loan date to prevent target leakage
pre_loan_trans = trans_loan[
    trans_loan['trans_date'] < trans_loan['loan_date']
].copy()
pre_loan_trans['days_before_loan'] = (
    pre_loan_trans['loan_date'] - pre_loan_trans['trans_date']
).dt.days

# =========================================================
# 3. Extract Core High-Signal Features Only
# =========================================================
# Pull base loan metadata directly from the clean 'loan' dataframe to avoid key errors
base_loans = loan.copy()
base_loans['loan_id'] = base_loans['loan_id'].astype(str)
base_loans.set_index('loan_id', inplace=True)

# Ensure target is integer
base_loans['target'] = base_loans['status'].isin(['B', 'D']).astype(int)

# Overall balance & history statistics from pre_loan_trans
balance_stats = pre_loan_trans.groupby('loan_id').agg(
    min_balance=('balance', 'min'),
    mean_balance=('balance', 'mean'),
    std_balance=('balance', 'std'),
    total_tx_count=('balance', 'count'),
    history_days=('days_before_loan', 'max'),
)

# Cash inflow vs outflow volumes
cashflow_totals = pre_loan_trans.pivot_table(
    index='loan_id',
    columns='type',
    values='amount_trans',
    aggfunc='sum',
    fill_value=0,
).rename(columns={'credit': 'total_inflow', 'withdrawal': 'total_outflow'})

# 90-day window
tx_90d = pre_loan_trans[pre_loan_trans['days_before_loan'] <= 90]
balance_90d = tx_90d.groupby('loan_id').agg(
    mean_balance_90d=('balance', 'mean')
)

# Overdraft frequency & Last balance before loan
neg_balance_series = (
    pre_loan_trans[pre_loan_trans['balance'] < 0]
    .groupby('loan_id')['trans_date']
    .nunique()
    .rename('freq_negative_balance')
)

last_balance_series = (
    pre_loan_trans.sort_values(['loan_id', 'trans_date'])
    .groupby('loan_id')['balance']
    .last()
    .rename('last_balance')
)

# Sanction interest (historical penalties)
sanction_series = (
    pre_loan_trans[pre_loan_trans['k_symbol'] == 'sanction_interest']
    .groupby('loan_id')['amount_trans']
    .sum()
    .rename('sanction_interest')
)

# Cash reliance ratio
income_ops = pre_loan_trans.pivot_table(
    index='loan_id',
    columns='operation',
    values='amount_trans',
    aggfunc='sum',
    fill_value=0,
)
salary_electronic = income_ops.get('collection_other_bank', 0)
cash_deposits = income_ops.get('credit_in_cash', 0)
cash_reliance_ratio = (cash_deposits) / (
    cash_deposits + salary_electronic + 1e-5
)
cash_reliance_ratio.name = 'cash_reliance_ratio'

# =========================================================
# 4. Merge & Calculate Key Ratios
# =========================================================
merged_trans = base_loans.join(
    [
        balance_stats,
        cashflow_totals,
        balance_90d,
        neg_balance_series,
        last_balance_series,
        sanction_series,
        cash_reliance_ratio,
    ]
).fillna(0)

merged_trans['months_history'] = (
    merged_trans['history_days'] / 30.4
).clip(lower=1)
merged_trans['avg_monthly_inflow'] = (
    merged_trans['total_inflow'] / merged_trans['months_history']
)
merged_trans['tx_per_month'] = (
    merged_trans['total_tx_count'] / merged_trans['months_history']
)

# Key Risk Ratios
merged_trans['dsti_ratio'] = merged_trans['payments'] / (
    merged_trans['avg_monthly_inflow'] + 1e-5
)
merged_trans['balance_trend_90d'] = merged_trans['mean_balance_90d'] / (
    merged_trans['mean_balance'] + 1e-5
)

# =========================================================
# 5. Export Streamlined Feature Store
# =========================================================
non_feature_cols = [
    'account_id',
    'date',
    'amount',
    'duration',
    'payments',
    'status',
    'target',
]
feature_cols = [c for c in merged_trans.columns if c not in non_feature_cols]

df_export = merged_trans[feature_cols].copy()
df_export.index = df_export.index.astype(str)
df_export.index.name = 'loan_id'

output_dir = '../modelData'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'features_transaction.csv')
df_export.to_csv(output_path)

print('=' * 50)
print(f'Saved streamlined transaction features to: {output_path}')
print(f'Rows (Loans):      {df_export.shape[0]}')
print(f'Feature Columns:   {df_export.shape[1]} (Reduced & Optimized)')
print(f'Missing Values:    {df_export.isna().sum().sum()}')
print('=' * 50)

df_export.head()

Saved streamlined transaction features to: ../modelData\features_transaction.csv
Rows (Loans):      682
Feature Columns:   19 (Reduced & Optimized)
Missing Values:    0


,is_closed,loan_date,min_balance,mean_balance,std_balance,total_tx_count,history_days,total_inflow,total_outflow,mean_balance_90d,freq_negative_balance,last_balance,sanction_interest,cash_reliance_ratio,months_history,avg_monthly_inflow,tx_per_month,dsti_ratio,balance_trend_90d
loan_id,,,,,,,,,,,,,,,,,,,
5314,1,1993-07-05,1100.0,12250.000000,8330.866301,4,105,20100.0,0.0,15966.666667,0.0,20100.0,0.0,1.000000,3.453947,5819.428571,1.158095,1.380376,1.303401
5316,1,1993-07-11,700.0,52083.859459,29122.059454,37,148,229893.7,177684.8,56036.575862,0.0,52208.9,0.0,1.000000,4.868421,47221.408649,7.600000,0.097625,1.075891
6863,1,1993-07-28,800.0,30060.954167,11520.184451,24,170,75146.0,54873.0,35160.929412,0.0,20272.8,0.0,0.060480,5.592105,13437.872941,4.291765,0.157614,1.169654
5325,1,1993-08-03,1000.0,41297.480000,14151.260443,25,185,120309.8,86017.2,46378.637500,0.0,34292.7,0.0,0.041001,6.085526,19769.826595,4.108108,0.148661,1.123038
7240,1,1993-09-06,600.0,57188.211111,25256.665817,27,204,276327.1,235214.0,60804.547059,0.0,41142.9,0.0,0.002180,6.710526,41178.156078,4.023529,0.111200,1.063236
